
# RoBERTa DAPT (MLM) on Lyrics with LoRA  
This notebook performs **domain-adaptive pretraining** of **RoBERTa** using a **Masked Language Modeling (MLM)** objective with **LoRA** for memory/time efficiency.


In [ ]:
pip install -U transformers datasets peft accelerate evaluate bitsandbytes


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# ==== Paths ====
TRAIN_PATH = '/content/drive/MyDrive/DALI/dali_lines/lyrics-train.csv'   
TEST_PATH  = '/content/drive/MyDrive/DALI/dali_lines/lyrics-test.csv'    #

# ==== Model & Training ====
MODEL_NAME = "roberta-base"    # or "roberta-large"
OUTPUT_DIR = "/content/drive/MyDrive/roberta/roberta-dapt-lyrics-lora"
SEQ_LEN = 256
MLM_PROB = 0.15

BATCH_SIZE = 16
GRAD_ACCUM = 2
NUM_EPOCHS = 12
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
SEED = 42

# Mixed precision + optimizer
USE_FP16 = True      
USE_8BIT_ADAM = True 

# ==== LoRA ====
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["query", "key", "value"]  # stable for RoBERTa attention blocks

# remove near-duplicates (per split) for cleaner signal
DROP_DUPLICATES = True
MIN_CHARS = 1    # optionally filter very short lines (set >1 to enable)


In [ ]:

import pandas as pd
import re

def load_split(path, drop_dupes=True, min_chars=1):
    df = pd.read_csv(path)
    if "text" not in df.columns:
        raise ValueError(f"{path} must contain a 'text' column.")
    s = (
        df["text"]
        .dropna()
        .astype(str)
        .map(lambda x: re.sub(r"\s+", " ", x).strip())
    )
    if min_chars and min_chars > 1:
        s = s[s.str.len() >= min_chars]
    if drop_dupes:
        s = s.drop_duplicates()
    return s.reset_index(drop=True)

train_texts = load_split(TRAIN_PATH, DROP_DUPLICATES, MIN_CHARS)
test_texts  = load_split(TEST_PATH,  DROP_DUPLICATES, MIN_CHARS)

print(f"Train texts: {len(train_texts):,}")
print(f"Test texts:  {len(test_texts):,}")
print('Example train line:', train_texts.iloc[0][:200] if len(train_texts) else "N/A")


In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
from functools import partial

train_ds = Dataset.from_pandas(pd.DataFrame({"text": train_texts}))
eval_ds  = Dataset.from_pandas(pd.DataFrame({"text": test_texts}))
ds = DatasetDict(train=train_ds, validation=eval_ds)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_function(batch, tokenizer, max_length):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_length,
        padding=False,
    )

tokenize_fn = partial(tokenize_function, tokenizer=tokenizer, max_length=SEQ_LEN)
tokenized = ds.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized


In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=MLM_PROB
)


In [ ]:
import torch
from transformers import AutoModelForMaskedLM
from peft import LoraConfig, get_peft_model, TaskType

torch.manual_seed(SEED)

base_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,  # <- use this for encoder-only MLM
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,          # ["query","key","value"] is fine
    bias="none",
)

model = get_peft_model(base_model, lora_config)

# Safety: explicitly mark as non-generative to avoid accidental .generate() calls
if hasattr(model, "can_generate"):
    model.can_generate = False

model.print_trainable_parameters()


In [ ]:

from transformers import TrainingArguments, Trainer
import torch

def get_precision_flags():
    if not torch.cuda.is_available():
        return False, False
    major, _ = torch.cuda.get_device_capability(0)
    bf16 = major >= 8  # Ampere+
    fp16 = (not bf16) and True
    return fp16 and USE_FP16, bf16 and USE_FP16

fp16_flag, bf16_flag = get_precision_flags()
optim_name = "paged_adamw_8bit" if USE_8BIT_ADAM else "adamw_torch"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=LEARNING_RATE,
    load_best_model_at_end=True,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    fp16=fp16_flag,
    bf16=bf16_flag,
    optim=optim_name,
    report_to="none",
    seed=SEED,
)

from transformers import EarlyStoppingCallback, Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # stop if 2 evals without improvement
)


In [ ]:

import math

train_result = trainer.train()
eval_metrics = trainer.evaluate()

print("Eval metrics:", eval_metrics)
ppl = math.exp(eval_metrics["eval_loss"]) if eval_metrics and "eval_loss" in eval_metrics else float("nan")
print(f"Perplexity (held-out test.csv): {ppl:.3f}")


In [ ]:

from peft import PeftModel
from transformers import AutoModelForMaskedLM

# Save LoRA adapter
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter to: {OUTPUT_DIR}")

